### NOTE: You're own baseball dataset will be needed to run this!! 

### Please use our DATA_PATH variable as a reference for our project setup. Thank you.

In [ ]:
# Load Libraries
import pandas as pd
import numpy as np
import seaborn as sns
import sklearn
import altair as alt
import matplotlib.pyplot as plt

In [ ]:
# Load People.csv, FieldingPost.csv, PitchingPost.csv, BattingPost.csv, Schools.csv, Salaries.csv, Appearances.csv, HallOfFame.csvf
DATA_PATH = "baseball/core/"
ppl  = pd.read_csv(DATA_PATH + "People.csv")
fld  = pd.read_csv(DATA_PATH + "FieldingPost.csv")
pch  = pd.read_csv(DATA_PATH + "PitchingPost.csv")
bat  = pd.read_csv(DATA_PATH + "BattingPost.csv")
scl  = pd.read_csv(DATA_PATH + "Schools.csv")
sal  = pd.read_csv(DATA_PATH + "Salaries.csv")
hof  = pd.read_csv(DATA_PATH + "HallOfFame.csv")
app  = pd.read_csv(DATA_PATH + "Appearances.csv")
inf  = pd.read_csv("infCoef.csv")

In [ ]:
# Check the datasets
print("People:", ppl.shape)
print("Salaries:", sal.shape)
print("Hall of Fame:", hof.shape)
print("Appearances:", app.shape)

In [ ]:
print("People columns")
print(ppl.columns.tolist())

print("\nSalary columns:")
print(sal.columns.tolist())

print("\nHall of Fame columns:")
print(hof.columns.tolist())

print("\nAppearance columns:")
print(app.columns.tolist())

In [ ]:
# ----------------------------------
# Analysis of Hall of Fame vs Salary
# ----------------------------------

# Particularly, is Hall of Fame membership associated with higher career salary among MLB players?
# Does the relationship remain when noting total seasons played?

In [ ]:
# We need to account for inflation in all salary calculations

salInf = sal.merge(
    inf,
    left_on="yearID",
    right_on="yearID",
    how="left"
)
display(salInf.head())
print("Missing inflation coefficients:", salInf["coefficient"].isna().sum())
salInf["salaryAdjusted"] = (
    salInf["salary"] * salInf["coefficient"]
)
display(
    salInf[
        ["playerID", "yearID", "salary", "coefficient", "salaryAdjusted"]
    ].head(20)
)

In [ ]:
# Identifying Hall of Fame players
hofPlayers = hof[hof["inducted"] == "Y"][["playerID"]].drop_duplicates()
print("Number of Hall of Fame players:", len(hofPlayers))
display(hofPlayers.head())

In [ ]:
# Lets check the salaries of the Hall of Famers

salaryHOF = salInf.merge(
    hofPlayers.assign(hallOfFame=True),
    on="playerID",
    how="left"
)
salaryHOF["hallOfFame"] = salaryHOF["hallOfFame"].fillna(False)
display(salaryHOF.head())

In [ ]:
# Calculate the total salary recorded

careerSalary = (
    salaryHOF
    .groupby(["playerID", "hallOfFame"], as_index=False)
    .agg(
        careerSalary=("salaryAdjusted", "sum"),
        averageSalary=("salaryAdjusted", "mean"),
        seasonsPaid=("yearID", "nunique")
    )
)

print("Rows to compare:", len(careerSalary))
careerSalary = careerSalary.merge(
    ppl[["playerID", "nameFirst", "nameLast"]],
    on="playerID",
    how="left"
)
careerSalary["name"] = (
    careerSalary["nameFirst"] + " " + careerSalary["nameLast"]
)
display(careerSalary.head())

In [ ]:
# Lets compare the groups
careerSalary.groupby("hallOfFame")[
    ["careerSalary", "averageSalary", "seasonsPaid"]
    ].median()

plt.figure(figsize=(8, 6))
sns.boxplot(
    data=careerSalary,
    x="hallOfFame",
    y="careerSalary"
)

plt.yscale("log")
plt.title("Career Salary of Hall of Fame and Non-Hall of Fame Players")
plt.xlabel("Hall of Fame Member")
plt.ylabel("Career Salary ($, logarithmic scale)")
plt.show()

In [ ]:
# The above graph shouldn't be all that surprising, so let's see if it's because they play longer or they actually command higher salaries

plt.figure(figsize=(10,7))
sns.scatterplot(
    data=careerSalary,
    x="seasonsPaid",
    y="careerSalary",
    hue="hallOfFame",
    alpha=0.6
)
plt.yscale("log")
plt.title("Career Salary vs. Number of Seasons")
plt.xlabel("Seasons with recorded salary")
plt.ylabel("Career Salary ($, logarithmic scale)")
plt.show()

In [ ]:
# ---------------------------------
# Analysis of Appearances vs Salary
# ---------------------------------

# Particularly, what is the relationship between player appearances and salary?
# Are there any players who earned unusually high salaries relative to their playing time?

In [ ]:
# Remembering that the column G_all notes total games played, we can see how many career appearances each player has made
careerAppearances = (
    app
    .groupby("playerID", as_index=False)
    .agg(
        careerGames=("G_all", "sum"),
        seasonsPlayed=("yearID", "nunique")
    )
)

display(careerAppearances.head())

In [ ]:
# Lets combine salary and appearances
salaryGames = careerSalary.merge(
    careerAppearances,
    on="playerID",
    how="inner"
)

display(salaryGames.head())

In [ ]:
# Salary per game analysis
salaryGames["salaryPerGame"] = (
    salaryGames["careerSalary"] / salaryGames["careerGames"]
)
salaryGames.sort_values(
    "salaryPerGame",
    ascending=False
)[
    [
        "name",
        "careerSalary",
        "careerGames",
        "salaryPerGame",
    ]
].head(20)

In [ ]:
# We will note some immediate outliers in the data, particularly the people with single digit games played.
# Injuries, releases, steroid use, and other weird seasons may create interesting outliers.
# We will keep these in mind as we make a general scatterplot for this.

plt.figure(figsize=(10,7))
sns.scatterplot(
    data=salaryGames,
    x="careerGames",
    y="careerSalary",
    alpha=0.5
)
plt.yscale("log")
plt.title("Career Games Played vs. Career Salary")
plt.xlabel("Career Games Played")
plt.ylabel("Career Salary ($, logarithmic scale)")
plt.show()

In [ ]:
# For fun, let's connect the Hall of Fame status
plt.figure(figsize=(10,7))
sns.scatterplot(
    data=salaryGames,
    x="careerGames",
    y="careerSalary",
    hue="hallOfFame",
    alpha=0.6
)

plt.yscale("log")
plt.title("Career Games vs. Career Salary by Hall of Fame Status")
plt.xlabel("Career Games Played")
plt.ylabel("Career Salary ($, logarithmic scale)")
plt.show()

In [ ]:
# Noting the above, we can use linear regression to find out who made a lot more money than expected given their career number of games.

from sklearn.linear_model import LinearRegression

# Using log salary because salaries are heavily skewedDATA_PATH
X = salaryGames[["careerGames"]]
y = np.log1p(salaryGames["careerSalary"])

model = LinearRegression()
model.fit(X,y)

salaryGames["predictedLogSalary"] = model.predict(X)

salaryGames["salaryResidual"] = (
    y - salaryGames["predictedLogSalary"]
)

outliers = salaryGames.sort_values(
    "salaryResidual",
    ascending=False
)

display(
    outliers[
        [
            "name",
            "careerGames",
            "careerSalary",
            "salaryPerGame",
            "salaryResidual"
        ]
    ].head(20)
)

plt.figure(figsize=(11,7))

sns.scatterplot(
    data=salaryGames,
    x="careerGames",
    y="careerSalary",
    alpha=0.35
)
topOutliers = salaryGames.nlargest(10, "salaryResidual")
display(topOutliers)
sns.scatterplot(
    data=topOutliers,
    x="careerGames",
    y="careerSalary",
    color="red",
    s=80,
    label="Top salary outliers"
)

plt.yscale("log")
plt.title("Career Games vs. Career Salary")
plt.xlabel("Career Games Played")
plt.ylabel("Career Salary ($, logarithmic scale)")
plt.legend()
plt.show()

In [ ]:
# Load People.csv, FieldingPost.csv, PitchingPost.csv, BattingPost.csv, Schools.csv, Salaries.csv, etc.
DATA_PATH = "baseball/core/"
ppl  = pd.read_csv(DATA_PATH + "People.csv")
fld  = pd.read_csv(DATA_PATH + "FieldingPost.csv")
pch  = pd.read_csv(DATA_PATH + "PitchingPost.csv")
bat  = pd.read_csv(DATA_PATH + "BattingPost.csv")
scl  = pd.read_csv(DATA_PATH + "Schools.csv")
sal  = pd.read_csv(DATA_PATH + "Salaries.csv")
col  = pd.read_csv(DATA_PATH + "CollegePlaying.csv")
infCoef = pd.read_csv("baseball/infCoef.csv")

In [ ]:
# "Clean" data
fld.drop(['CS','SB','PB'], axis="columns", inplace=True)
ppl.drop(['birthMonth','birthDay','birthCountry',
       'birthState','birthCity','deathYear','deathMonth','deathDay',
       'deathCountry','deathState','deathCity'], axis="columns", inplace=True)
ppl['finalGame'] = ppl.finalGame.astype('datetime64[ns]')
ppl = ppl[ppl.finalGame.dt.year > 1900] # modern baseball rules established in 1901
sal = sal[sal.yearID > 1900]

# Join data
college = col.merge(scl, on='schoolID', how='left').iloc[:, :-1]

# Apply coef on everyones salary
sal = sal.merge(infCoef, on="yearID", how="left")
sal["finalSalary"] = sal.coefficient * sal.salary

In [ ]:
last_college = college.groupby('playerID').max()
last_college.reset_index(inplace=True)
last_college = last_college[last_college.yearID > 1984]

In [ ]:
sorta_ivy = ['Brown University', 'Columbia University', 'Cornell University',
             'Dartmouth College', 'Harvard University', 'University of Pennsylvania',
             'Princeton University', 'Yale University', 'Northwestern University',
             'University of Notre Dame', 'Georgetown University',
             'Johns Hopkins University', 'University of Chicago', 'Tufts University',
             'Massachusetts Institute of Technology']  

sporty = ['University of Southern California', 'Louisiana State University', 'Texas A&M University',
          'University of Texas at Austin', 'University of Arizona', 'Arizona State University',
          'University of Oklahoma', 'Oklahoma State University', 'University of Miami', 'Florida State University',
          'University of Florida', 'University of Georgia', 'Mississippi State University', 'University of Mississippi',
          'Auburn University', 'University of South Carolina', 'Clemson University',
          'University of California, Los Angeles',
          'California State University Fullerton', 'California State University Long Beach',
          'California State University Fresno',
          'Wichita State University', 'Pepperdine University', 'Baylor University',
          'University of North Carolina at Chapel Hill', 'Oregon State University', 'Coastal Carolina University',
          'Stanford University', 'Vanderbilt University', 'Duke University', 'Rice University']

last_college = last_college.assign(
    school_type = lambda x: np.select(
        [x.name_full.isin(sorta_ivy), x.name_full.isin(sporty)],
        [1, 2],
        default=0
    )
)

In [ ]:
player_salary = sal.groupby("playerID").agg(
    career_total = ("finalSalary", "sum"),
    peak_salary  = ("finalSalary", "max"),
    avg_salary   = ("finalSalary", "mean"),
    years_played = ("finalSalary", "count")
).reset_index()

analysis_df = player_salary.merge(
    last_college[["playerID", "school_type"]],
    on="playerID", how="inner"
)

label_map = {0: "Others", 1: "Academics", 2: "Sport Schools"}
analysis_df["school_type_label"] = analysis_df["school_type"].map(label_map)

In [ ]:
avg_chart = alt.Chart(analysis_df).mark_boxplot().encode(
    x=alt.X("school_type_label:N", title="School Type", sort=["Others", "Academics", "Sport Schools"]),
    y=alt.Y(
        "avg_salary:Q",
        scale=alt.Scale(type="log"),
        axis=alt.Axis(format="$,.0f", title="Career Average Salary ($, log scale)")
    )
).properties(
    width=200,
    title='How Well Were Players Paid Across Career?'
)

peak_chart = alt.Chart(analysis_df).mark_boxplot().encode(
    x=alt.X("school_type_label:N", title="School Type", sort=["Others", "Academics", "Sport Schools"]),
    y=alt.Y(
        "peak_salary:Q",
        scale=alt.Scale(type="log"),
        axis=alt.Axis(format="$,.0f", title="Career Peak Salary ($, log scale)")
    )
).properties(
    width=200,
    title='How Well Were Players Paid At Their Best?'
)

years_chart = alt.Chart(analysis_df).mark_boxplot().encode(
    x=alt.X("school_type_label:N", title="School Type", sort=["Others", "Academics", "Sport Schools"]),
    y=alt.Y(
        "years_played:Q",
        axis=alt.Axis(title="Career Years Played")
    )
).properties(
    width=200,
    title='How Long Did Players Play?'
)

combined_chart = avg_chart | peak_chart | years_chart
combined_chart

In [ ]:
# Look at salary over time 
stat_sal = pd.DataFrame()

# Take the summary stats
stat_sal['Mean']          = sal.groupby("yearID")["finalSalary"].mean()
stat_sal['Median']        = sal.groupby("yearID")["finalSalary"].median()
stat_sal['.25 Quartile']  = sal.groupby("yearID")["finalSalary"].quantile(0.25)
stat_sal['.75 Quartile']  = sal.groupby("yearID")["finalSalary"].quantile(0.75)
stat_sal.reset_index(inplace=True)
stat_sal = stat_sal.melt(id_vars="yearID")

# NOTE: Salary only starts at 1985 (womp-womp)

# Plot trend
sal_chart = alt.Chart(stat_sal).mark_line(interpolate="monotone").encode(
    x=alt.X(
        "yearID:Q",
        title='Year',
        axis=alt.Axis(labelAngle=-45, format='d'),
        scale=alt.Scale(domain=[1985, stat_sal.yearID.max()])
    ),
    y=alt.Y(
        "value:Q",
        title='Salary ($)',
        axis=alt.Axis(format='$,.0f')
        ),
    color=alt.Color(
    "variable:N",
    sort=["Mean", "Median", ".25 Quartile", ".75 Quartile"],
    legend=alt.Legend(title='Summary Stat.')
    )
).properties(
    title={
        'text':'MLB Players Salary Trends',
        'subtitle':'Salaries are rising'
    }
)

sal_chart = sal_chart.configure_title(
    fontSize=18,
    anchor='start'
)

In [ ]:
sal_chart.show()